In [ ]:
!pip -q install datasets torchmetrics[image] torch-fidelity

In [ ]:
from pathlib import Path
import sys
import os
import json
import time
import copy
import random
from dataclasses import dataclass, asdict
from itertools import product

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from torchvision.utils import make_grid

from torchmetrics.image.fid import FrechetInceptionDistance
from torchmetrics.image.kid import KernelInceptionDistance

from tqdm.auto import tqdm

In [ ]:
SEED = 42

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Paths Projeto

In [ ]:
ROOT = Path("/content/drive/MyDrive/IAG")
DATA_ROOT = ROOT / "archive"
SRC_ROOT = ROOT / "src"
RESULTS_ROOT = ROOT / "results"

MODELS_DIR = RESULTS_ROOT / "models"
IMAGES_DIR = RESULTS_ROOT / "images"
METRICS_DIR = RESULTS_ROOT / "metrics"

MODELS_DIR.mkdir(parents=True, exist_ok=True)
IMAGES_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)

TRAINING_CSV_PATH = DATA_ROOT / "training_20_percent.csv"

print("ROOT:", ROOT)
print("DATA_ROOT:", DATA_ROOT)
print("SRC_ROOT:", SRC_ROOT)
print("TRAINING_CSV_PATH:", TRAINING_CSV_PATH)

# 2. Dataset

## 2.1 Loader (professor)

In [ ]:
sys.path.append(str(SRC_ROOT))
from artbench_local_dataset import load_kaggle_artbench10_splits

## 2.2 Carregar dataset

In [ ]:
hf_ds = load_kaggle_artbench10_splits(DATA_ROOT)
train_hf = hf_ds["train"]
test_hf = hf_ds["test"]

print("Train size:", len(train_hf))
print("Test size :", len(test_hf))
print("Columns   :", train_hf.column_names)

label_feature = train_hf.features["label"]
class_names = list(label_feature.names)
num_classes = len(class_names)
print("Num classes:", num_classes)
print("Class names:", class_names)

## 2.3 Helpers dataset

In [ ]:
def load_ids_from_training_csv(csv_path, index_column="train_id_original"):
    df = pd.read_csv(csv_path)
    if index_column not in df.columns:
        raise ValueError(f"Coluna '{index_column}' não encontrada no CSV.")
    return df[index_column].astype(int).tolist()

def safe_num_workers(requested_workers=2):
    return 0  # estabilidade no Colab

class HFDatasetTorch(Dataset):
    def __init__(self, hf_dataset, transform=None, indices=None):
        self.hf_dataset = hf_dataset
        self.transform = transform
        self.indices = indices if indices is not None else list(range(len(hf_dataset)))

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        idx = int(self.indices[i])
        sample = self.hf_dataset[idx]
        img = sample["image"]
        label = int(sample["label"])

        if self.transform is not None:
            img = self.transform(img)

        return img, label, idx

## 2.4 Transform

In [ ]:
IMAGE_SIZE = 32
BATCH_SIZE = 64
NUM_WORKERS = safe_num_workers(2)

transform = T.Compose([
    T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    T.ToTensor(),
    T.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

## 2.5 Construir subset (20%)

In [ ]:
INDEX_COLUMN = "train_id_original"
VAL_FRACTION = 0.2

train_ids = load_ids_from_training_csv(TRAINING_CSV_PATH, index_column=INDEX_COLUMN)

rng = np.random.default_rng(SEED)
train_ids = np.array(train_ids)
rng.shuffle(train_ids)

num_val = int(len(train_ids) * VAL_FRACTION)
val_indices = train_ids[:num_val].tolist()
subset_train_indices = train_ids[num_val:].tolist()

subset_train_ds = HFDatasetTorch(train_hf, transform=transform, indices=subset_train_indices)
subset_val_ds   = HFDatasetTorch(train_hf, transform=transform, indices=val_indices)

full_train_ds   = HFDatasetTorch(train_hf, transform=transform)
full_test_ds    = HFDatasetTorch(test_hf, transform=transform)

print("Subset train dataset length:", len(subset_train_ds))
print("Subset val dataset length  :", len(subset_val_ds))
print("Full train dataset length  :", len(full_train_ds))
print("Full test dataset length   :", len(full_test_ds))

# 3. FID e KID

In [ ]:
def denorm(x):
    return ((x + 1) / 2).clamp(0, 1)

def to_uint8_images(x):
    x = denorm(x).clamp(0, 1)
    x = (x * 255.0).round().to(torch.uint8)
    return x

def build_metric_loader(dataset, batch_size=128):
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
    )

subset_val_metric_loader = build_metric_loader(subset_val_ds, batch_size=128)
full_test_metric_loader = build_metric_loader(full_test_ds, batch_size=128)

@torch.no_grad()
def evaluate_generator_fid_kid(generator, real_loader, latent_dim, n_samples=5000, device=DEVICE):
    generator.eval()

    fid = FrechetInceptionDistance(feature=2048, normalize=False).to(device)
    kid = KernelInceptionDistance(subset_size=100, normalize=False).to(device)

    # reais
    real_count = 0
    for real_images, _, _ in real_loader:
        real_images = real_images.to(device)
        real_uint8 = to_uint8_images(real_images)

        needed = min(real_uint8.size(0), n_samples - real_count)
        if needed <= 0:
            break

        fid.update(real_uint8[:needed], real=True)
        kid.update(real_uint8[:needed], real=True)

        real_count += needed
        if real_count >= n_samples:
            break

    # falsas
    fake_count = 0
    batch_size = real_loader.batch_size if real_loader.batch_size is not None else 64

    while fake_count < n_samples:
        cur_bsz = min(batch_size, n_samples - fake_count)
        z = torch.randn(cur_bsz, latent_dim, 1, 1, device=device)
        fake_images = generator(z)
        fake_uint8 = to_uint8_images(fake_images)

        fid.update(fake_uint8, real=False)
        kid.update(fake_uint8, real=False)
        fake_count += cur_bsz

    fid_score = float(fid.compute().detach().cpu())
    kid_mean, kid_std = kid.compute()

    return {
        "fid": fid_score,
        "kid_mean": float(kid_mean.detach().cpu()),
        "kid_std": float(kid_std.detach().cpu()),
    }

@torch.no_grad()
def quick_eval(generator, latent_dim, real_loader, device=DEVICE, n_samples=1000):
    return evaluate_generator_fid_kid(
        generator=generator,
        real_loader=real_loader,
        latent_dim=latent_dim,
        n_samples=n_samples,
        device=device,
    )

# 4. Modelo WGAN-GP

In [ ]:
class PixelNorm(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x, eps=1e-8):
        return x / torch.sqrt(torch.mean(x**2, dim=1, keepdim=True) + eps)

class Generator(nn.Module):
    def __init__(self, latent_dim=128, g_features=64, use_pixelnorm=True):
        super().__init__()

        def g_block(in_ch, out_ch, k, s, p, use_pn=True):
            layers = [
                nn.ConvTranspose2d(in_ch, out_ch, k, s, p, bias=False),
                nn.BatchNorm2d(out_ch),
            ]
            if use_pn and use_pixelnorm:
                layers.append(PixelNorm())
            layers.append(nn.ReLU(True))
            return nn.Sequential(*layers)

        self.net = nn.Sequential(
            g_block(latent_dim, g_features * 8, 4, 1, 0),
            g_block(g_features * 8, g_features * 4, 4, 2, 1),
            g_block(g_features * 4, g_features * 2, 4, 2, 1),
            nn.ConvTranspose2d(g_features * 2, 3, 4, 2, 1, bias=False),
            nn.Tanh()
        )

    def forward(self, z):
        return self.net(z)

class Critic(nn.Module):
    def __init__(self, d_features=64):
        super().__init__()

        self.net = nn.Sequential(
            nn.Conv2d(3, d_features, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(d_features, d_features * 2, 4, 2, 1, bias=False),
            nn.InstanceNorm2d(d_features * 2, affine=True),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(d_features * 2, d_features * 4, 4, 2, 1, bias=False),
            nn.InstanceNorm2d(d_features * 4, affine=True),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(d_features * 4, 1, 4, 1, 0, bias=False),
        )

    def forward(self, x):
        return self.net(x).view(-1)


@dataclass
class WGANGPConfig:
    model_name: str = "wgangp"
    latent_dim: int = 128
    g_features: int = 64
    d_features: int = 64
    batch_size: int = 64
    epochs: int = 100
    lr_g: float = 1e-4
    lr_d: float = 1e-4
    beta1: float = 0.0
    beta2: float = 0.9
    lambda_gp: float = 10.0
    n_critic: int = 5
    use_pixelnorm: bool = True
    seed: int = 42
    save_every: int = 10


def weights_init(m):
    classname = m.__class__.__name__
    if classname.find('Conv') != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find('BatchNorm') != -1 or classname.find('InstanceNorm') != -1:
        if hasattr(m, 'weight') and m.weight is not None:
            nn.init.normal_(m.weight.data, 1.0, 0.02)
        if hasattr(m, 'bias') and m.bias is not None:
            nn.init.constant_(m.bias.data, 0)

def save_generated_grid(fake_images, epoch, run_name, out_dir, nrow=8, show=False):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    fake_images = denorm(fake_images.detach().cpu())
    grid = make_grid(fake_images, nrow=nrow, padding=2)
    np_img = grid.permute(1, 2, 0).numpy()

    plt.figure(figsize=(8, 8))
    plt.imshow(np.clip(np_img, 0, 1))
    plt.axis("off")
    plt.title(f"{run_name} - epoch {epoch}")

    save_path = out_dir / f"{run_name}_epoch_{epoch:03d}.png"
    plt.savefig(save_path, bbox_inches="tight")
    if show:
        plt.show()
    plt.close()

    return save_path

def gradient_penalty(critic, real, fake, device=DEVICE):
    bsz = real.size(0)
    alpha = torch.rand(bsz, 1, 1, 1, device=device)
    interpolated = alpha * real + (1 - alpha) * fake
    interpolated.requires_grad_(True)

    mixed_scores = critic(interpolated)

    grad_outputs = torch.ones_like(mixed_scores, device=device)

    gradients = torch.autograd.grad(
        inputs=interpolated,
        outputs=mixed_scores,
        grad_outputs=grad_outputs,
        create_graph=True,
        retain_graph=True,
        only_inputs=True,
    )[0]

    gradients = gradients.view(gradients.size(0), -1)
    gp = ((gradients.norm(2, dim=1) - 1) ** 2).mean()
    return gp

class EarlyStopper:
    def __init__(self, patience=4, min_delta=0.5):
        self.patience = patience
        self.min_delta = min_delta
        self.best_score = None
        self.bad_rounds = 0

    def step(self, score):
        if self.best_score is None:
            self.best_score = score
            self.bad_rounds = 0
            return False, True

        if score < self.best_score - self.min_delta:
            self.best_score = score
            self.bad_rounds = 0
            return False, True

        self.bad_rounds += 1
        return self.bad_rounds >= self.patience, False

## 4.1 Treino

In [ ]:
def train_wgangp_with_early_stopping(
    cfg,
    train_loader,
    val_loader_metric,
    quick_eval_samples=1000,
    eval_every=5,
    patience=4,
    min_delta=0.5,
):
    seed_everything(cfg.seed)

    netG = Generator(
        latent_dim=cfg.latent_dim,
        g_features=cfg.g_features,
        use_pixelnorm=cfg.use_pixelnorm
    ).to(DEVICE)

    netC = Critic(d_features=cfg.d_features).to(DEVICE)

    netG.apply(weights_init)
    netC.apply(weights_init)

    optimizerG = optim.Adam(netG.parameters(), lr=cfg.lr_g, betas=(cfg.beta1, cfg.beta2))
    optimizerC = optim.Adam(netC.parameters(), lr=cfg.lr_d, betas=(cfg.beta1, cfg.beta2))

    run_name = (
        f"{cfg.model_name}"
        f"_ld{cfg.latent_dim}"
        f"_gf{cfg.g_features}"
        f"_df{cfg.d_features}"
        f"_bs{cfg.batch_size}"
        f"_lrg{cfg.lr_g}"
        f"_lrd{cfg.lr_d}"
        f"_b1{cfg.beta1}"
        f"_b2{cfg.beta2}"
        f"_gp{cfg.lambda_gp}"
        f"_nc{cfg.n_critic}"
        f"_pn{int(cfg.use_pixelnorm)}"
        f"_seed{cfg.seed}"
    )

    history = {
        "g_loss": [],
        "c_loss": [],
        "wasserstein": [],
        "gp": [],
        "quick_fid": [],
        "quick_kid_mean": [],
        "quick_kid_std": [],
        "eval_epochs": [],
    }

    stopper = EarlyStopper(patience=patience, min_delta=min_delta)

    best_fid = float("inf")
    best_epoch = 0
    best_state_G = None
    best_state_C = None

    fixed_noise_local = torch.randn(64, cfg.latent_dim, 1, 1, device=DEVICE)

    print(f"\n===== TRAINING {run_name} =====")

    for epoch in range(1, cfg.epochs + 1):
        start_epoch = time.time()

        netG.train()
        netC.train()

        epoch_g_loss = 0.0
        epoch_c_loss = 0.0
        epoch_wass = 0.0
        epoch_gp = 0.0

        for real_images, _, _ in tqdm(train_loader, leave=False, desc=f"Epoch {epoch}/{cfg.epochs}"):
            real_images = real_images.to(DEVICE)
            bsz = real_images.size(0)

            critic_loss_total = 0.0
            critic_wass_total = 0.0
            critic_gp_total = 0.0

            for _ in range(cfg.n_critic):
                z = torch.randn(bsz, cfg.latent_dim, 1, 1, device=DEVICE)
                fake_images = netG(z)

                critic_real = netC(real_images)
                critic_fake = netC(fake_images.detach())

                wasserstein = critic_real.mean() - critic_fake.mean()
                gp = gradient_penalty(netC, real_images, fake_images.detach(), device=DEVICE)

                loss_C = -wasserstein + cfg.lambda_gp * gp

                optimizerC.zero_grad()
                loss_C.backward()
                optimizerC.step()

                critic_loss_total += loss_C.item()
                critic_wass_total += wasserstein.item()
                critic_gp_total += gp.item()

            z = torch.randn(bsz, cfg.latent_dim, 1, 1, device=DEVICE)
            fake_images = netG(z)
            loss_G = -netC(fake_images).mean()

            optimizerG.zero_grad()
            loss_G.backward()
            optimizerG.step()

            epoch_c_loss += critic_loss_total / cfg.n_critic
            epoch_wass += critic_wass_total / cfg.n_critic
            epoch_gp += critic_gp_total / cfg.n_critic
            epoch_g_loss += loss_G.item()

        epoch_c_loss /= len(train_loader)
        epoch_wass /= len(train_loader)
        epoch_gp /= len(train_loader)
        epoch_g_loss /= len(train_loader)

        history["c_loss"].append(epoch_c_loss)
        history["wasserstein"].append(epoch_wass)
        history["gp"].append(epoch_gp)
        history["g_loss"].append(epoch_g_loss)

        elapsed = time.time() - start_epoch
        print(
            f"Epoch [{epoch:03d}/{cfg.epochs}] | "
            f"C Loss: {epoch_c_loss:.4f} | G Loss: {epoch_g_loss:.4f} | "
            f"W: {epoch_wass:.4f} | GP: {epoch_gp:.4f} | Time: {elapsed:.1f}s"
        )

        if epoch % cfg.save_every == 0 or epoch == 1 or epoch == cfg.epochs:
            netG.eval()
            with torch.no_grad():
                fake_fixed = netG(fixed_noise_local)
            save_generated_grid(fake_fixed, epoch, run_name, out_dir=IMAGES_DIR, show=False)

        if epoch % eval_every == 0 or epoch == cfg.epochs:
            quick_metrics = quick_eval(
                generator=netG,
                latent_dim=cfg.latent_dim,
                real_loader=val_loader_metric,
                device=DEVICE,
                n_samples=quick_eval_samples,
            )

            qfid = quick_metrics["fid"]
            qkid = quick_metrics["kid_mean"]
            qkid_std = quick_metrics["kid_std"]

            history["quick_fid"].append(qfid)
            history["quick_kid_mean"].append(qkid)
            history["quick_kid_std"].append(qkid_std)
            history["eval_epochs"].append(epoch)

            print(
                f"  -> Quick Eval @ epoch {epoch}: "
                f"FID={qfid:.3f}, KID={qkid:.5f} ± {qkid_std:.5f}"
            )

            should_stop, improved = stopper.step(qfid)

            if improved:
                best_fid = qfid
                best_epoch = epoch
                best_state_G = copy.deepcopy(netG.state_dict())
                best_state_C = copy.deepcopy(netC.state_dict())

            if should_stop:
                print(f"  -> Early stopping triggered at epoch {epoch}")
                break

    if best_state_G is None:
        best_state_G = copy.deepcopy(netG.state_dict())
        best_state_C = copy.deepcopy(netC.state_dict())
        best_epoch = cfg.epochs

    netG.load_state_dict(best_state_G)
    netC.load_state_dict(best_state_C)

    model_path_G = MODELS_DIR / f"{run_name}_BEST_G.pt"
    model_path_C = MODELS_DIR / f"{run_name}_BEST_C.pt"

    torch.save(netG.state_dict(), model_path_G)
    torch.save(netC.state_dict(), model_path_C)

    print(f"Best epoch: {best_epoch} | Best quick FID: {best_fid:.3f}")
    print("Saved generator:", model_path_G)
    print("Saved critic   :", model_path_C)

    return netG, netC, history, run_name, best_epoch, best_fid

## 4.3 Grid search no subset (20%)

In [ ]:

grid_space = {
    "latent_dim": [96, 128],
    "g_features": [64],
    "d_features": [64],
    "batch_size": [64],
    "epochs": [100],
    "lr_g": [1e-4, 2e-4],
    "lr_d": [1e-4],
    "beta1": [0.0],
    "beta2": [0.9],
    "lambda_gp": [10.0],
    "n_critic": [3, 5],
    "use_pixelnorm": [True],
    "save_every": [10],
}

def build_grid_configs(grid_space, seed=42, model_name="wgangp_grid_subset"):
    keys = list(grid_space.keys())
    values = list(grid_space.values())
    configs = []

    for combo in product(*values):
        kwargs = dict(zip(keys, combo))
        cfg = WGANGPConfig(
            model_name=model_name,
            seed=seed,
            **kwargs
        )
        configs.append(cfg)

    return configs

grid_configs = build_grid_configs(grid_space, seed=42)
print("Número de configs da grid:", len(grid_configs))
for i, cfg in enumerate(grid_configs[:5], 1):
    print(i, cfg)

In [ ]:
grid_results = []

for cfg in grid_configs:
    print("\n" + "=" * 100)
    print("GRID RUN")
    print(cfg)
    print("=" * 100)

    train_loader = DataLoader(
        subset_train_ds,
        batch_size=cfg.batch_size,
        shuffle=True,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
    )

    netG, netC, history, run_name, best_epoch, best_quick_fid = train_wgangp_with_early_stopping(
        cfg=cfg,
        train_loader=train_loader,
        val_loader_metric=subset_val_metric_loader,
        quick_eval_samples=1000,
        eval_every=5,
        patience=4,
        min_delta=0.5,
    )

    final_metrics = evaluate_generator_fid_kid(
        generator=netG,
        real_loader=subset_val_metric_loader,
        latent_dim=cfg.latent_dim,
        n_samples=5000,
        device=DEVICE
    )

    row = asdict(cfg)
    row.update({
        "run_name": run_name,
        "best_epoch": best_epoch,
        "best_quick_fid": best_quick_fid,
        "final_subset_fid": final_metrics["fid"],
        "final_subset_kid_mean": final_metrics["kid_mean"],
        "final_subset_kid_std": final_metrics["kid_std"],
    })
    grid_results.append(row)

    history_path = METRICS_DIR / f"{run_name}_history.json"
    with open(history_path, "w") as f:
        json.dump(history, f, indent=2)

    del netG, netC
    torch.cuda.empty_cache()

grid_df = pd.DataFrame(grid_results)
grid_csv_path = METRICS_DIR / "wgangp_grid_subset_results.csv"
grid_df.to_csv(grid_csv_path, index=False)

print("Saved:", grid_csv_path)
display(grid_df)

## 4.4 Melhores Configs

In [ ]:
# raking

grid_summary = (
    grid_df
    .sort_values(by=["final_subset_fid", "final_subset_kid_mean"], ascending=[True, True])
    .reset_index(drop=True)
)

grid_summary_path = METRICS_DIR / "wgangp_grid_subset_ranking.csv"
grid_summary.to_csv(grid_summary_path, index=False)

print("Saved:", grid_summary_path)
display(grid_summary.head())

In [ ]:
# top 3
top_k = 3
top_grid_df = grid_summary.head(top_k).copy()
display(top_grid_df)

In [ ]:
def row_to_wgangp_config(row):
    return WGANGPConfig(
        model_name=row["model_name"],
        latent_dim=int(row["latent_dim"]),
        g_features=int(row["g_features"]),
        d_features=int(row["d_features"]),
        batch_size=int(row["batch_size"]),
        epochs=int(row["epochs"]),
        lr_g=float(row["lr_g"]),
        lr_d=float(row["lr_d"]),
        beta1=float(row["beta1"]),
        beta2=float(row["beta2"]),
        lambda_gp=float(row["lambda_gp"]),
        n_critic=int(row["n_critic"]),
        use_pixelnorm=bool(row["use_pixelnorm"]),
        seed=42,
        save_every=int(row["save_every"]),
    )

top_configs = [row_to_wgangp_config(row) for _, row in top_grid_df.iterrows()]
for cfg in top_configs:
    print(cfg)

In [ ]:
top_configs_json = METRICS_DIR / "wgangp_top3_configs.json"

with open(top_configs_json, "w") as f:
    json.dump(top_grid_df.to_dict(orient="records"), f, indent=2)

In [ ]:
top_configs_json = METRICS_DIR / "wgangp_top3_configs.json"

with open(top_configs_json, "r") as f:
    top_configs_dict = json.load(f)

def row_to_wgan_config(row):
    return WGANConfig(
        model_name=row["model_name"],
        latent_dim=int(row["latent_dim"]),
        g_features=int(row["g_features"]),
        d_features=int(row["d_features"]),
        batch_size=int(row["batch_size"]),
        epochs=int(row["epochs"]),
        lr_g=float(row["lr_g"]),
        lr_d=float(row["lr_d"]),
        beta1=float(row["beta1"]),
        beta2=float(row["beta2"]),
        lambda_gp=float(row["lambda_gp"]),
        n_critic=int(row["n_critic"]),
        use_pixelnorm=bool(row["use_pixelnorm"]),
        seed=42,
        save_every=int(row["save_every"]),
    )

top_configs = [row_to_wgan_config(cfg) for cfg in top_configs_dict]

In [ ]:
for cfg in top_configs:
    print(cfg)

## 4.5 Top 3 configs × 10 seeds (subset)

In [ ]:
selection_seeds = [42, 123, 999, 2024, 7, 11, 21, 33, 77, 101]

top_results = []

for base_cfg in top_configs:
    for seed in selection_seeds:
        cfg = copy.deepcopy(base_cfg)
        cfg.seed = seed
        cfg.model_name = "wgangp_top10seeds_subset"

        print("\n" + "=" * 100)
        print("TOP CONFIG × SEED")
        print(cfg)
        print("=" * 100)

        train_loader = DataLoader(
            subset_train_ds,
            batch_size=cfg.batch_size,
            shuffle=True,
            num_workers=0,
            pin_memory=torch.cuda.is_available(),
        )

        netG, netC, history, run_name, best_epoch, best_quick_fid = train_wgangp_with_early_stopping(
            cfg=cfg,
            train_loader=train_loader,
            val_loader_metric=subset_val_metric_loader,
            quick_eval_samples=1000,
            eval_every=5,
            patience=4,
            min_delta=0.5,
        )

        final_metrics = evaluate_generator_fid_kid(
            generator=netG,
            real_loader=subset_val_metric_loader,
            latent_dim=cfg.latent_dim,
            n_samples=5000,
            device=DEVICE
        )

        row = asdict(cfg)
        row.update({
            "run_name": run_name,
            "best_epoch": best_epoch,
            "best_quick_fid": best_quick_fid,
            "final_subset_fid": final_metrics["fid"],
            "final_subset_kid_mean": final_metrics["kid_mean"],
            "final_subset_kid_std": final_metrics["kid_std"],
        })
        top_results.append(row)

        history_path = METRICS_DIR / f"{run_name}_history.json"
        with open(history_path, "w") as f:
            json.dump(history, f, indent=2)

        del netG, netC
        torch.cuda.empty_cache()

top_results_df = pd.DataFrame(top_results)
top_results_csv = METRICS_DIR / "wgangp_topconfigs_10seeds_subset.csv"
top_results_df.to_csv(top_results_csv, index=False)

print("Saved:", top_results_csv)
display(top_results_df.head())

In [ ]:
group_cols = [
    "latent_dim", "g_features", "d_features", "batch_size",
    "lr_g", "lr_d", "beta1", "beta2", "lambda_gp", "n_critic",
    "use_pixelnorm"
]

top_summary = (
    top_results_df
    .groupby(group_cols, dropna=False)
    .agg(
        fid_mean=("final_subset_fid", "mean"),
        fid_std=("final_subset_fid", "std"),
        kid_mean=("final_subset_kid_mean", "mean"),
        kid_std=("final_subset_kid_mean", "std"),
        best_epoch_mean=("best_epoch", "mean"),
    )
    .reset_index()
    .sort_values(by="fid_mean", ascending=True)
)

top_summary_csv = METRICS_DIR / "wgangp_topconfigs_10seeds_summary.csv"
top_summary.to_csv(top_summary_csv, index=False)

print("Saved:", top_summary_csv)
display(top_summary)

## 5. Daset completo - melhor configuração com 10 seeds

In [ ]:
best_row = top_summary.iloc[0]

best_full_cfg = WGANGPConfig(
    model_name="wgangp_full_final",
    latent_dim=int(best_row["latent_dim"]),
    g_features=int(best_row["g_features"]),
    d_features=int(best_row["d_features"]),
    batch_size=int(best_row["batch_size"]),
    epochs=100,
    lr_g=float(best_row["lr_g"]),
    lr_d=float(best_row["lr_d"]),
    beta1=float(best_row["beta1"]),
    beta2=float(best_row["beta2"]),
    lambda_gp=float(best_row["lambda_gp"]),
    n_critic=int(best_row["n_critic"]),
    use_pixelnorm=bool(best_row["use_pixelnorm"]),
    seed=42,
    save_every=10,
)

final_seeds = [42, 123, 999, 2024, 7, 11, 21, 33, 77, 101]
print(best_full_cfg)
print("Seeds:", final_seeds)

In [ ]:
full_train_loader = DataLoader(
    full_train_ds,
    batch_size=best_full_cfg.batch_size,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

full_test_metric_loader = DataLoader(
    full_test_ds,
    batch_size=128,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

In [ ]:
def load_completed_seeds(json_path):
    if json_path.exists():
        with open(json_path, "r") as f:
            return set(json.load(f))
    return set()

def save_completed_seeds(json_path, completed_seeds):
    with open(json_path, "w") as f:
        json.dump(sorted(list(completed_seeds)), f, indent=2)

def load_existing_results(csv_path):
    if csv_path.exists():
        return pd.read_csv(csv_path)
    return pd.DataFrame()

In [ ]:
full_results_csv = METRICS_DIR / "wgangp_full_final_10seeds.csv"
full_summary_csv = METRICS_DIR / "wgangp_full_final_10seeds_summary.csv"
completed_seeds_json = METRICS_DIR / "wgangp_full_final_completed_seeds.json"

completed_seeds = load_completed_seeds(completed_seeds_json)
existing_df = load_existing_results(full_results_csv)

if not existing_df.empty:
    print(f"Já existem {len(existing_df)} resultados guardados.")
else:
    print("Ainda não existem resultados guardados.")

print("Seeds já concluídos:", completed_seeds)

for seed in final_seeds:
    if seed in completed_seeds:
        print(f"\n[SKIP] Seed {seed} já concluída.")
        continue

    cfg = copy.deepcopy(best_full_cfg)
    cfg.seed = seed

    print("\n" + "=" * 100)
    print(f"FULL DATASET RUN | seed={seed}")
    print(cfg)
    print("=" * 100)

    try:
        netG, netC, history, run_name, best_epoch, best_quick_fid = train_wgangp_with_early_stopping(
            cfg=cfg,
            train_loader=full_train_loader,
            val_loader_metric=full_test_metric_loader,
            quick_eval_samples=1000,
            eval_every=5,
            patience=4,
            min_delta=0.5,
        )

        final_metrics = evaluate_generator_fid_kid(
            generator=netG,
            real_loader=full_test_metric_loader,
            latent_dim=cfg.latent_dim,
            n_samples=5000,
            device=DEVICE
        )

        row = asdict(cfg)
        row.update({
            "run_name": run_name,
            "best_epoch": best_epoch,
            "best_quick_fid": best_quick_fid,
            "final_full_fid": final_metrics["fid"],
            "final_full_kid_mean": final_metrics["kid_mean"],
            "final_full_kid_std": final_metrics["kid_std"],
        })

        if existing_df.empty:
            existing_df = pd.DataFrame([row])
        else:
            existing_df = pd.concat([existing_df, pd.DataFrame([row])], ignore_index=True)

        history_path = METRICS_DIR / f"{run_name}_history.json"
        with open(history_path, "w") as f:
            json.dump(history, f, indent=2)

        existing_df.to_csv(full_results_csv, index=False)

        completed_seeds.add(seed)
        save_completed_seeds(completed_seeds_json, completed_seeds)

        print(f"[OK] Seed {seed} concluída e guardada.")
        print(f"Current results saved to: {full_results_csv}")

        del netG, netC
        torch.cuda.empty_cache()

    except Exception as e:
        print(f"[ERROR] Seed {seed} falhou: {e}")
        torch.cuda.empty_cache()
        break

In [ ]:
full_results_csv = METRICS_DIR / "wgangp_full_final_10seeds.csv"
full_summary_csv = METRICS_DIR / "wgangp_full_final_10seeds_summary.csv"
completed_seeds_json = METRICS_DIR / "wgangp_full_final_completed_seeds.json"

def load_completed_seeds(json_path):
    if json_path.exists():
        with open(json_path, "r") as f:
            return set(json.load(f))
    return set()

def save_completed_seeds(json_path, completed_seeds):
    with open(json_path, "w") as f:
        json.dump(sorted(list(completed_seeds)), f, indent=2)

def load_existing_results(csv_path):
    if csv_path.exists():
        return pd.read_csv(csv_path)
    return pd.DataFrame()

completed_seeds = load_completed_seeds(completed_seeds_json)
existing_df = load_existing_results(full_results_csv)

print("Seeds concluídos:", completed_seeds)
print("N resultados já guardados:", len(existing_df))

In [ ]:
completed_seeds = load_completed_seeds(completed_seeds_json)
completed_seeds.add(77)
save_completed_seeds(completed_seeds_json, completed_seeds)
print(completed_seeds)

In [ ]:
from dataclasses import dataclass

@dataclass
class WGANGPConfig:
    model_name: str
    latent_dim: int
    g_features: int
    d_features: int
    batch_size: int
    epochs: int
    lr_g: float
    lr_d: float
    beta1: float
    beta2: float
    lambda_gp: float
    n_critic: int
    use_pixelnorm: bool
    seed: int
    save_every: int


best_full_cfg = WGANGPConfig(
    model_name="wgangp_full_final",
    latent_dim=128,
    g_features=64,
    d_features=64,
    batch_size=64,
    epochs=100,
    lr_g=1e-4,
    lr_d=1e-4,
    beta1=0.0,
    beta2=0.9,
    lambda_gp=10.0,
    n_critic=3,
    use_pixelnorm=True,
    seed=42,  # vai ser substituído no loop
    save_every=10,
)

In [ ]:
missing_seeds = [101]

for seed in missing_seeds:
    if seed in completed_seeds:
        print(f"[SKIP] Seed {seed} já concluída.")
        continue

    cfg = copy.deepcopy(best_full_cfg)
    cfg.seed = seed

    print("\n" + "=" * 100)
    print(f"FULL DATASET RUN | seed={seed}")
    print(cfg)
    print("=" * 100)

    try:
        netG, netC, history, run_name, best_epoch, best_quick_fid = train_wgangp_with_early_stopping(
            cfg=cfg,
            train_loader=full_train_loader,
            val_loader_metric=full_test_metric_loader,
            quick_eval_samples=1000,
            eval_every=5,
            patience=4,
            min_delta=0.5,
        )

        final_metrics = evaluate_generator_fid_kid(
            generator=netG,
            real_loader=full_test_metric_loader,
            latent_dim=cfg.latent_dim,
            n_samples=5000,
            device=DEVICE
        )

        row = asdict(cfg)
        row.update({
            "run_name": run_name,
            "best_epoch": best_epoch,
            "best_quick_fid": best_quick_fid,
            "final_full_fid": final_metrics["fid"],
            "final_full_kid_mean": final_metrics["kid_mean"],
            "final_full_kid_std": final_metrics["kid_std"],
        })

        if existing_df.empty:
            existing_df = pd.DataFrame([row])
        else:
            existing_df = pd.concat([existing_df, pd.DataFrame([row])], ignore_index=True)

        history_path = METRICS_DIR / f"{run_name}_history.json"
        with open(history_path, "w") as f:
            json.dump(history, f, indent=2)

        existing_df.to_csv(full_results_csv, index=False)

        completed_seeds.add(seed)
        save_completed_seeds(completed_seeds_json, completed_seeds)

        print(f"[OK] Seed {seed} concluída e guardada.")
        print(f"Current results saved to: {full_results_csv}")

        del netG, netC
        torch.cuda.empty_cache()

    except Exception as e:
        print(f"[ERROR] Seed {seed} falhou: {e}")
        torch.cuda.empty_cache()
        continue

In [ ]:
full_results_df = pd.read_csv(full_results_csv)
display(full_results_df)

full_summary_df = pd.DataFrame([{
    "latent_dim": full_results_df["latent_dim"].iloc[0],
    "g_features": full_results_df["g_features"].iloc[0],
    "d_features": full_results_df["d_features"].iloc[0],
    "batch_size": full_results_df["batch_size"].iloc[0],
    "lr_g": full_results_df["lr_g"].iloc[0],
    "lr_d": full_results_df["lr_d"].iloc[0],
    "beta1": full_results_df["beta1"].iloc[0],
    "beta2": full_results_df["beta2"].iloc[0],
    "lambda_gp": full_results_df["lambda_gp"].iloc[0],
    "n_critic": full_results_df["n_critic"].iloc[0],
    "use_pixelnorm": full_results_df["use_pixelnorm"].iloc[0],
    "fid_mean": full_results_df["final_full_fid"].mean(),
    "fid_std": full_results_df["final_full_fid"].std(),
    "kid_mean": full_results_df["final_full_kid_mean"].mean(),
    "kid_std": full_results_df["final_full_kid_mean"].std(),
    "best_epoch_mean": full_results_df["best_epoch"].mean(),
    "num_completed_seeds": len(full_results_df),
}])

full_summary_df.to_csv(full_summary_csv, index=False)

print("Saved:", full_summary_csv)
display(full_summary_df)

## 5.1 Visualização imagens

In [ ]:
def get_random_real_images(dataset, n=10):
    indices = torch.randperm(len(dataset))[:n]
    imgs = [dataset[i][0] for i in indices]
    return torch.stack(imgs)

def generate_fake_images(generator, latent_dim, n, device):
    generator.eval()
    with torch.no_grad():
        z = torch.randn(n, latent_dim, 1, 1, device=device)
        fake = generator(z)
    return denorm(fake.cpu())

def plot_block(real_imgs, fake_imgs, block_id):
    n = real_imgs.shape[0]
    fig, axes = plt.subplots(2, n, figsize=(20, 4))

    for i in range(n):
        axes[0, i].imshow(real_imgs[i].permute(1, 2, 0))
        axes[0, i].axis("off")

    for i in range(n):
        axes[1, i].imshow(fake_imgs[i].permute(1, 2, 0))
        axes[1, i].axis("off")

    axes[0, 0].set_ylabel("Real", fontsize=12)
    axes[1, 0].set_ylabel("Generated", fontsize=12)

    plt.suptitle(f"Block {block_id}", fontsize=14)
    plt.tight_layout()
    plt.show()

best_seed_row = full_results_df.sort_values("final_full_fid").iloc[0]
best_model_path = MODELS_DIR / f"{best_seed_row['run_name']}_BEST_G.pt"

netG = Generator(
    latent_dim=int(best_seed_row["latent_dim"]),
    g_features=int(best_seed_row["g_features"]),
    use_pixelnorm=bool(best_seed_row["use_pixelnorm"]),
).to(DEVICE)

state_dict = torch.load(best_model_path, map_location=DEVICE)
netG.load_state_dict(state_dict)
netG.eval()

latent_dim_best = int(best_seed_row["latent_dim"])

for block in range(4):
    real_imgs = denorm(get_random_real_images(full_train_ds, n=10))
    fake_imgs = generate_fake_images(netG, latent_dim_best, n=10, device=DEVICE)
    plot_block(real_imgs, fake_imgs, block + 1)